# Course notebook — Cloud and Enterprise AI Architecture

Design secure, governed, resilient AI platforms and map them to managed services without losing vendor-neutral reasoning.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Practice case · Recovery day at a public utility

Fatima designs a policy assistant for a utility. Field staff sometimes work with a poor connection. The legal team needs auditable access and the operations team needs a safe recovery path if a managed service or deployment is wrong. Her architecture review starts with those needs and then considers cloud capabilities.

The workplace scenario is fictional.


## Architecture and trust boundaries


### Lecture 01 · Enterprise AI Building Blocks

Decompose platforms into gateways, model access, retrieval, tool integration, identity, policy, observability, evaluation, and developer experience.

#### Start with the capabilities the crew needs

A field worker must view current procedures on a weak connection and know which revision they have. The design review starts by comparing three vendor product names.

Draw data, identity, retrieval, and recovery boundaries first. Then map capabilities to products. The weak-connection case may call for a timestamped local view, regardless of which platform serves the assistant.

#### Read the mechanism

**AI gateways**

AI gateways centralize model access with auth, rate limits, logging, routing, and policy enforcement for enterprise teams.

All Bedrock and OpenAI calls flow through gateway applying PII scrub and budget caps.

**model catalog**

Model catalog lists approved models with risk tier, eval status, and allowed use cases for developers.

Catalog shows gpt-4o approved tier-2; llama-local approved tier-1 air-gapped only.

**shared retrieval**

Shared retrieval services provide governed indexes, embedding pipelines, and search APIs reused across products.

Enterprise policy index serves HR bot and IT bot with tenant filters from one platform team.

**tool registry**

Tool registry catalogs approved agent tools with schemas, owners, and security review status.

Registry entry for create_jira_ticket includes schema v2 and pentest date.

**platform engineering**

Platform engineering builds self-service AI infrastructure—gateways, eval harnesses, templates—so product teams ship faster safely.

Platform provides RAG starter kit with auth, ingest, eval wired to corporate SSO.

#### Worked example

**Situation:** An architect must implement the same governed AI capability on different cloud providers without rewriting product logic each migration.

**Baseline:** Vendor-specific SDK calls scattered through application code.

**Application:** Draw logical platform: gateway, model catalog, shared retrieval, tool registry, identity, policy, observability, eval service—name products only after capabilities mapped.

**Test cases:** (1) Normal: swap model provider behind gateway. (2) Boundary: shared retrieval ACL model portable. (3) Adversarial: leaky abstraction hiding provider limits (context size).

**Measurement:** Portability score (# provider-locked calls), time to map architecture on second cloud.

**Design question:** Which capability boundary must stay stable across vendors?

**Predict before running.** Stable capability boundaries make vendor choices replaceable and governance consistent. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 12.1: Enterprise AI Building Blocks"""

CHAPTER = "12.1"
print("chapter hook:", CHAPTER)
capabilities = ["gateway", "retrieval", "tool registry", "identity", "observability"]
products = {"aws": "bedrock", "azure": "foundry", "gcp": "vertex"}
for cap in capabilities:
    print(cap, "maps to provider-specific service behind interface")
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Draw a logical platform architecture before naming products.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why define logical capabilities before products?
2. What makes vendor choices replaceable?
3. What architecture baseline embeds SDKs in app code?

<details><summary>Check your reasoning</summary>

Q1: Capabilities survive vendor renames and mergers. Q2: Stable interfaces and owned data. Q3: Direct Bedrock/OpenAI calls everywhere.

</details>


### Lecture 02 · Identity, Data, and Trust Boundaries

Apply authentication, authorization, tenancy, secrets, encryption, residency, lineage, and audit to AI data flows.

#### Permissions need to travel with the data

A shared policy index serves managers and contractors. A contractor’s result should never depend on a policy document they cannot read.

Propagate caller, role, and source access into retrieval and testing. Check the same question for distinct roles and verify logs and caches keep the boundary too.

#### Read the mechanism

**identity**

Identity establishes who users and services are—SSO, service principals, workload identity—for AI data access.

Employee SSO identity flows to retrieval filters and audit logs on every query.

**authorization**

Authorization ensures retrieved and acted-upon data respects user permissions—not just authentication. RAG without authZ leaks restricted documents into answers.

An employee should not retrieve executive compensation docs via semantic search without role checks.

**multi-tenancy**

Multi-tenancy isolates customer data, indexes, quotas, and configs in shared AI platforms.

Tenant A embeddings never appear in Tenant B vector search results.

**data residency**

Data residency restricts processing and storage to approved geographic regions for legal compliance.

EU customer prompts and indexes stay in eu-west inference and storage only.

**audit**

Audit logs record who accessed which models, documents, and tools with immutable retention for compliance.

Log entry: user, query hash, retrieved doc IDs, model version, timestamp.

#### Worked example

**Situation:** Enterprise assistant crosses HR data, IT tickets, and manager identity—auditors ask who can see what at each hop.

**Baseline:** Single shared API key to all backend services.

**Application:** Threat-model flows with authentication, authorization, tenancy, encryption in transit/at rest, residency, lineage, audit logs on model and retrieval calls.

**Test cases:** (1) Normal: employee accesses own onboarding status. (2) Boundary: manager views direct report. (3) Adversarial: confused deputy via tool using admin credentials.

**Measurement:** Unauthorized access test pass rate, audit log completeness, data residency compliance checklist.

**Design question:** Where must authorization occur—in the model prompt or at each tool/data boundary?

**Predict before running.** A model call does not suspend ordinary identity and data-security requirements. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 12.2: Identity, Data, and Trust Boundaries"""

CHAPTER = "12.2"
print("chapter hook:", CHAPTER)
boundaries = ["user->gateway", "gateway->retrieval", "tool->HR API"]
for b in boundaries:
    print(b, "requires authZ check")
print("inspect step", 1)
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Threat-model an enterprise assistant across trust boundaries.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why doesn't a model call suspend security requirements?
2. How does tenancy differ from authentication?
3. What trust baseline uses one global API key?

<details><summary>Check your reasoning</summary>

Q1: Models do not enforce policy; services must. Q2: AuthN proves identity; tenancy scopes data access. Q3: Shared key with no per-user scopes.

</details>


**Section checkpoint.** Sketch the logical model, search, compute, and identity capabilities before picking services.


## AWS and Azure implementations


### Lecture 03 · AWS Managed AI

Map foundation models, ML lifecycle, retrieval, serverless compute, containers, workflow, identity, storage, and monitoring to AWS services.

#### Plan a dependency failure before choosing a managed stack

One embedding service falls behind during a reindex. Field staff could still use approved keyword search to locate procedures.

Tie each managed dependency to a service boundary and test a reduced-capability path. Compare recovery and observability as part of platform selection, alongside cost and ease of provisioning.

#### Read the mechanism

**Amazon Bedrock**

Amazon Bedrock provides managed access to foundation models from multiple providers via unified AWS APIs with IAM integration and private networking.

Invoke Claude and Titan through Bedrock in VPC without exposing keys on laptops.

**SageMaker**

Amazon SageMaker covers ML training, tuning, hosting, and monitoring for custom and foundation models on AWS.

Fine-tune and deploy custom classifier on SageMaker endpoint with autoscaling and Model Monitor.

**OpenSearch**

Amazon OpenSearch supports lexical, vector, and hybrid search with k-NN indexes for RAG on AWS.

OpenSearch k-NN index stores policy embeddings filtered by IAM-scoped document metadata.

**Lambda and EKS**

AWS Lambda and EKS provide serverless functions and Kubernetes clusters for AI orchestration, agents, and custom servers.

Lambda handles lightweight ingest triggers; EKS hosts vLLM GPU workloads with Karpenter scaling.

**CloudWatch and IAM**

CloudWatch and IAM deliver AWS monitoring, alerting, and access control for AI workloads—metrics, logs, roles, policies.

IAM role grants Bedrock invoke only; CloudWatch alarm on 5xx rate triggers runbook.

#### Worked example

**Situation:** Organization standardizes on AWS; team maps enterprise RAG design to Bedrock, OpenSearch, Lambda/EKS, IAM, CloudWatch.

**Baseline:** Lift-and-shift generic diagram with wrong service couplings.

**Application:** Map each logical component to AWS managed services, estimate trade-offs (ops vs control), document IAM boundaries, cost drivers, and gaps needing custom code.

**Test cases:** (1) Normal: Bedrock invoke with IAM role. (2) Boundary: OpenSearch Serverless vs managed cluster for ACL needs. (3) Adversarial: overly broad IAM policy on Lambda retriever.

**Measurement:** Architecture review score, IAM least-privilege pass, estimated monthly cost at 1M queries.

**Design question:** When would EKS beat Lambda for the tool execution layer?

**Predict before running.** Start with logical capabilities; use managed services where their constraints match the system. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 12.3: AWS Managed AI"""

CHAPTER = "12.3"
print("chapter hook:", CHAPTER)
mapping = {"models": "Bedrock", "search": "OpenSearch", "compute": "Lambda/EKS", "identity": "IAM"}
for cap, svc in mapping.items():
    print(f"{cap} -> {svc}")
print("inspect step", 1)
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Map the enterprise RAG design to AWS and estimate managed-service trade-offs.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why start from logical design before naming AWS products?
2. What trade do managed services impose?
3. What cloud baseline copies tutorial architecture unchanged?

<details><summary>Check your reasoning</summary>

Q1: Prevents forcing problems into familiar services. Q2: Less ops, more coupling and constraint. Q3: Default serverless RAG blog with no ACL model.

</details>


### Lecture 04 · Azure Managed AI

Map models, ML, search, functions, containers, integration, identity, data, security, and operations to Azure.

#### Review the person behind each role assignment

An identity integration grants broad access to an operations group that includes short-term vendors. The demo accounts all belonged to permanent staff.

Test real role categories and lifecycle transitions, such as an expiring assignment. Ensure emergency access is auditable and revocable before trusting any platform’s default group mapping.

#### Read the mechanism

**Azure AI Foundry**

Azure AI Foundry is Microsoft's unified portal for model deployment, fine-tuning, evaluation, and agent tooling integrated with Azure services.

Deploy GPT-4o mini, run eval flow, and promote to managed endpoint from Foundry pipeline.

**Azure OpenAI**

Azure OpenAI Service hosts OpenAI models in Azure regions with private networking, content filters, and Entra ID auth.

Enterprise chatbot calls gpt-4o in tenant VNet with content safety filters enabled.

**Azure AI Search**

Azure AI Search provides hybrid lexical-vector search, semantic ranker, and skill pipelines for RAG on Azure.

Indexer pipeline OCRs PDFs, enriches metadata, and indexes vectors for copilot retrieval.

**AKS and Functions**

AKS and Azure Functions run containerized model servers and event-driven AI glue code on Azure.

Function triggers on blob upload; AKS serves GPU embedding model with HPA.

**Entra ID and Monitor**

Microsoft Entra ID and Azure Monitor provide identity, RBAC, and observability for Azure AI workloads.

Entra groups map to AI Search index ACLs; Monitor alerts on token spike anomalies.

#### Worked example

**Situation:** Same RAG design must map to Azure for a subsidiary already on Entra ID and Azure AI Search.

**Baseline:** Assume Azure equals AWS with different names.

**Application:** Map to Azure AI Foundry/OpenAI, AI Search, Functions/AKS, Entra ID, Monitor; compare identity integration advantages and coupling risks vs AWS map.

**Test cases:** (1) Normal: Entra-scoped search index. (2) Boundary: hybrid search skill configuration. (3) Adversarial: guest account excessive search permissions.

**Measurement:** Identity integration effort score, feature parity gaps list, migration effort from AWS map.

**Design question:** Which Azure-native integration most reduces custom auth code—and what coupling does it create?

**Predict before running.** Cloud-native integration can accelerate governance but increases platform coupling. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 12.4: Azure Managed AI"""

CHAPTER = "12.4"
print("chapter hook:", CHAPTER)
integrations = {"identity": "Entra ID", "search": "Azure AI Search", "models": "Azure OpenAI"}
benefit = "group-based ACL on indexes"
coupling = "Entra-specific token claims"
print({"benefit": benefit, "coupling": coupling})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Map the same RAG design to Azure and compare identity integration.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. How can cloud-native integration accelerate governance?
2. When does platform coupling become risky?
3. What multi-cloud baseline ignores identity differences?

<details><summary>Check your reasoning</summary>

Q1: Uses existing directory groups for retrieval ACLs. Q2: Harder migration when IdP-specific claims embed in logic. Q3: Same IAM assumptions on every cloud.

</details>


**Section checkpoint.** Compare the AWS and Azure mapping. Identify which identity and data controls require verification.


## Portability and operating model


### Lecture 05 · Google Cloud and Portable Patterns

Map Vertex AI, search, Cloud Run, GKE, data, events, identity, and operations while identifying portable seams.

#### Portability includes export and restore

The organization can export prompts and documents, but its indexes, evaluations, and permission maps live in provider-specific formats.

Decide which artifacts must be portable and test their restore path. Migration risk is visible only when a second deployment reproduces the relevant behavior and access boundaries.

#### Read the mechanism

**Vertex AI**

Google Vertex AI offers unified model training, tuning, deployment, and evaluation on GCP with Gemini and open models.

Fine-tune Gemini on proprietary data and deploy to private endpoint with VPC-SC.

**Vertex AI Search**

Vertex AI Search (Discovery Engine) provides enterprise search and grounding APIs with document ingest and ranking.

Ingest GCS policy PDFs; grounding API returns answers with source references.

**Cloud Run and GKE**

Cloud Run and GKE deploy serverless containers and Kubernetes GPU workloads on Google Cloud.

Cloud Run serves CPU embedding API; GKE Autopilot runs LLM inference with TPU/GPU node pools.

**Cloud IAM**

Google Cloud IAM binds roles to identities for least-privilege access to Vertex, Storage, and BigQuery in AI pipelines.

Service account invokes Vertex prediction only; humans cannot read raw training bucket.

**portable interfaces**

Portable interfaces—OpenAI-compatible APIs, OTel traces, standard embedding dims—reduce lock-in across clouds.

Gateway speaks OpenAI schema; backends swap Bedrock, Azure, or vLLM without client changes.

#### Worked example

**Situation:** Global corp wants Google Cloud option for one region while keeping portable core elsewhere.

**Baseline:** Lowest-common-denominator design avoiding all managed features—shipping nothing.

**Application:** Map to Vertex AI, Vertex AI Search, Cloud Run/GKE, Cloud IAM; mark portable seams (OpenAPI gateway, OIDC, exportable embeddings index) vs GCP-specific optimizations.

**Test cases:** (1) Normal: Cloud Run tool service behind portable gateway. (2) Boundary: Vertex feature unavailable in region. (3) Adversarial: proprietary index format blocking migration.

**Measurement:** Portable interface count, migration drill time to alternate cloud stub.

**Design question:** Which seam is worth duplicating to preserve data ownership?

**Predict before running.** Portability is achieved through deliberate contracts and data ownership, not lowest-common-denominator design. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 12.5: Google Cloud and Portable Patterns"""

CHAPTER = "12.5"
print("chapter hook:", CHAPTER)
portable = ["OpenAPI gateway", "OIDC auth", "Parquet export of embeddings"]
gcp_specific = ["Vertex native grounding API"]
print({"portable": portable, "avoid_lockin": len(gcp_specific) == 0 or True})
print("inspect step", 1)
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Map the design to Google Cloud and identify the migration boundary.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. How is portability achieved deliberately?
2. What is wrong with lowest-common-denominator portability?
3. What portability baseline avoids all managed AI?

<details><summary>Check your reasoning</summary>

Q1: Contracts, owned data, swappable adapters—not weakest design. Q2: It sacrifices needed features without real exit path. Q3: Self-host everything with no SLA plan.

</details>


### Lecture 06 · Enterprise Operating Model

Define platform teams, product teams, centers of enablement, governance, service catalogs, SLOs, chargeback, vendor management, and adoption.

#### Write the runbook for the person on duty

An upstream service goes down during a weekend. Both teams think the other team owns approval to roll back.

Name who detects, who decides, and who verifies recovery for each dependency. Practice the handoff so a real outage does not become the first test of team ownership.

#### Read the mechanism

**team topology**

Team topology assigns platform, product, and enabling teams for AI delivery with clear interaction modes.

Platform team owns gateway; product teams own prompts and evals within guardrails.

**service catalog**

Service catalog lists internal AI products—approved models, RAG templates, tools—for self-service discovery.

Developer portal shows tier-2 chatbot template with cost estimate and onboarding steps.

**SLOs**

SLOs define target reliability and latency—availability, p95 latency, eval faithfulness—for AI platform services.

Gateway SLO: 99.9% availability, p95 <2s excluding model provider outages.

**FinOps**

FinOps tracks and optimizes AI spend—tokens, GPU hours, API fees—against business value.

Dashboard shows cost per successful ticket deflection by model route.

**vendor management**

Vendor management evaluates AI providers on security, compliance, cost, performance, and exit strategy.

Annual review of OpenAI/Azure/Bedrock DPAs, data retention, and failover plan.

#### Worked example

**Situation:** AI platform rollout fails because product teams bypass central services; leadership wants operating model clarity.

**Baseline:** Ad hoc "AI champions" with no ownership or funding model.

**Application:** Define platform vs product team responsibilities, service catalog, SLOs, FinOps chargeback, vendor management, enablement cadence, roadmap tied to adoption metrics.

**Test cases:** (1) Normal: team consumes model gateway. (2) Boundary: exception for regulated experiment. (3) Adversarial: shadow stack duplicates retrieval without security review.

**Measurement:** Catalog adoption %, platform SLO attainment, shadow IT incidents.

**Design question:** What incentive aligns product teams with central platform without blocking innovation?

**Predict before running.** Architecture succeeds only when ownership, incentives, operations, and delivery practices align. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 12.6: Enterprise Operating Model"""

CHAPTER = "12.6"
print("chapter hook:", CHAPTER)
RACI = {"platform": "operate gateway", "product": "own use-case evals", "governance": "tier approvals"}
for role, duty in RACI.items():
    print(role, duty)
print("inspect step", 1)
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Create a responsibility matrix and platform roadmap.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why does architecture fail without operating model alignment?
2. What belongs in a platform service catalog?
3. What enterprise baseline is tools-only with no ownership?

<details><summary>Check your reasoning</summary>

Q1: Unclear ownership → bypass, inconsistency, incidents. Q2: SLOs, APIs, support tier, cost model. Q3: Purchased licenses with no shared platform team.

</details>


**Section checkpoint.** Map the same design to Google Cloud and assign an owner for each operational control.


## Course workshop

Compare the logical keys in the provider maps. Identify a capability that would need its own abstraction and conformance test.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Keep logical capabilities separate from provider mappings."""
CAPABILITIES = {
    "model_access": {"aws": "Bedrock", "azure": "Azure AI Foundry", "gcp": "Vertex AI"},
    "retrieval": {"aws": "OpenSearch", "azure": "Azure AI Search", "gcp": "Vertex AI Search"},
    "identity": {"aws": "IAM", "azure": "Entra ID", "gcp": "Cloud IAM"},
}


def architecture(provider):
    return {capability: services[provider] for capability, services in CAPABILITIES.items()}


for provider in ("aws", "azure", "gcp"):
    print(provider, architecture(provider))


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
maps = [architecture(provider) for provider in ("aws", "azure", "gcp")]
assert set(maps[0]) == set(maps[1]) == set(maps[2])
assert maps[0] != maps[1]
print('Workshop checks passed')


## Course project

Produce a multi-cloud-capable enterprise AI reference architecture and five architecture decision records.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
